# Caso de estudio Reto 2

##José Miguel Tobo Yepes

La empresa Monstermash, dedicada al desarrollo de juegos en línea, busca predecir qué jugadores tienen mayor probabilidad de realizar compras dentro de la aplicación. Para abordar este problema, se empleará un modelo de árbol de decisión que utilice como variables de entrada Age (edad del cliente), Income (ingresos del hogar), Years (años de experiencia usando juegos en línea) y Hours (horas de juego semanales). Estas variables fueron seleccionadas por su relevancia directa con el comportamiento de compra: la edad permite identificar patrones generacionales, los ingresos reflejan la capacidad económica del jugador, la experiencia en juegos en línea está asociada con el grado de compromiso con la actividad, y las horas de juego semanales indican el nivel de interacción con la plataforma. Con estos atributos, el modelo busca capturar patrones que permitan estimar con mayor precisión la probabilidad de compra.

Llevar a cabo la implementación de un modelo de árbol de decisión que permita predecir qué jugadores es probable que hagan compras dentro de la aplicación.
variables:
+ Age
+ Income: Ingresos anuales del hogar
+ Years (Años usando juegos en línea) Tipo: Numérica continua (años). Descripción: Tiempo que el cliente ha usado juegos en línea. Mayor tiempo puede indicar más familiaridad y compromiso con la plataforma.
+ Hours (Horas de juego por semana) Tipo: Numérica continua (horas). Descripción: Cantidad de horas que el jugador dedica semanalmente a juegos en línea. Más horas podrían correlacionarse con mayor interacción y posibilidad de compra.
+ Buy (Variable de salida)

In [21]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier #Modelo de arbol de decisión
from sklearn.metrics import confusion_matrix
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


#1. Cargar los datos

In [22]:
nxl=('/content/drive/MyDrive/BI /2. BD2_In_App_Pur_Int.xlsx')

In [23]:
  XDB=pd.read_excel(nxl,sheet_name=0)#Base de datos AAA
  XDB=XDB.dropna()
  XDB.head()

,Age,Sex,Income,Years,Hours,CreditCard,Facebook,Buy
0,40,0,139,3,25,1,1,0
1,26,0,138,7,4,1,1,0
2,48,0,138,7,8,1,1,0
3,60,0,138,6,29,0,1,0
4,21,0,137,5,21,0,1,0


In [24]:
#Seleccionamos las variables de interes (Entrada y Salida)
XDB=XDB[['Age','Income','Years','Hours','CreditCard','Buy']]
XDB.head()

,Age,Income,Years,Hours,CreditCard,Buy
0,40,139,3,25,1,0
1,26,138,7,4,1,0
2,48,138,7,8,1,0
3,60,138,6,29,0,0
4,21,137,5,21,0,0


In [25]:
#Separar variables de entrada y salida
#Entrada se van a llamar XD
XD=XDB[['Age','Income','Years','Hours']] #Entrada
yd=XDB[['Buy']] #Salida o de referencia

#2. Implementar el modelo de Arbol de Decision

In [26]:
mar=DecisionTreeClassifier(criterion='gini',max_depth=4) #max_depth es la maxima profundidad del arbol (son el numero de variables de entrada que tenemos)
mar.fit(XD,yd)

DecisionTreeClassifier(max_depth=4)

In [38]:
#Para la grafica del Arbol
from sklearn.tree import export_graphviz
from pydotplus import graph_from_dot_data
dot_data=export_graphviz(mar,feature_names=XD.columns,filled=True,rounded=True)
graph=graph_from_dot_data(dot_data)
graph.write_png('arbol_Reto2.png')

True

#3. Se evalua el comportamiento del modelo frente a los datos

In [28]:
ydp=mar.predict(XD) #backtesting
cm=confusion_matrix(yd,ydp)
print(cm)
VN=cm[0,0];FP=cm[0,1];FN=cm[1,0];VP=cm[1,1]
print('VN=',VN,'FP=',FP,'FN=',FN,'VP=',VP)

[[139  50]
 [ 11 200]]
VN= 139 FP= 50 FN= 11 VP= 200


Segun esta matriz de confusion el modelo esta funcionando relativamente bien


In [29]:
Ter=(FP+FN)/(FP+FN+VN+VP)
print('Tasa de error=',Ter)# Comportamiento general
Ex=(VN+VP)/(FP+FN+VN+VP)
print('Exactitud=',Ex)#Comportamiento erroneo del modelo
Sen=VP/(VP+FN)
print('Sensibilidad=',Sen)#Comportamiento positivo del modelo
Esp=VN/(VN+FP)
print('Especificidad=',Esp)#Comportamiento frente a los prenegativos del modelo
Prec=VP/(VP+FP)
print('Precision=',Prec)#Comportamiento positivo del modelo
PredNeg=VN/(VN+FN)
print('Predictividad negativa=',PredNeg)#Comportamiento negativo del modelo

Tasa de error= 0.1525
Exactitud= 0.8475
Sensibilidad= 0.9478672985781991
Especificidad= 0.7354497354497355
Precision= 0.8
Predictividad negativa= 0.9266666666666666


In [40]:
# Crear una base de datos con los datos que me proporciona el excel
Prueba_2_jugadores = pd.DataFrame({
    'Age': [21, 50],
    'Income': [128, 128],
    'Years': [3, 1],
    'Hours': [22, 6]
})

# Usar el modelo ya entrenado
probabilities = mar.predict_proba(Prueba_2_jugadores)

print(f"Probabilidad para el primer jugador (Age: 21, Income: 128, Years: 3, Hours: 22): {probabilities[0][1]:.2f}")
print(f"Probabilidad para el segundo jugador (Age: 50, Income: 128, Years: 1, Hours: 6): {probabilities[1][1]:.2f}")

Probabilidad para el primer jugador (Age: 21, Income: 128, Years: 3, Hours: 22): 0.67
Probabilidad para el segundo jugador (Age: 50, Income: 128, Years: 1, Hours: 6): 0.00


#Nodos puros

In [31]:
import numpy as np

def reglas_puras(modelo, feature_names):
    tree = modelo.tree_
    reglas = []

    def recorre(nodo, condiciones):
        left, right = tree.children_left[nodo], tree.children_right[nodo]
        # es hoja
        if left == -1 and right == -1:
            counts = tree.value[nodo][0]
            clases_no_cero = np.count_nonzero(counts)
            impurity = tree.impurity[nodo]
            if impurity == 0.0 or clases_no_cero == 1:
                reglas.append({
                    "regla": " AND ".join(condiciones) if condiciones else "(sin condiciones)",
                    "muestras": int(counts.sum()),
                    "dist": counts.astype(int).tolist()
                })
            return

        feat = feature_names[tree.feature[nodo]]
        thr = tree.threshold[nodo]

        # Rama izquierda: <= umbral
        recorre(left, condiciones + [f"{feat} <= {thr:.2f}"])
        # Rama derecha: > umbral
        recorre(right, condiciones + [f"{feat} > {thr:.2f}"])

    recorre(0, [])
    return reglas

rp = reglas_puras(mar, list(XD.columns))
print("Nodos puros:")
for r in rp:
    print(f"- {r['regla']} | muestras={r['muestras']} | dist={r['dist']}")

Nodos puros:
- Income <= 63.50 | muestras=1 | dist=[1, 0]
- Income > 63.50 AND Years <= 1.50 AND Age <= 23.00 AND Years > 0.50 | muestras=1 | dist=[1, 0]
- Income > 63.50 AND Years <= 1.50 AND Age > 23.00 | muestras=1 | dist=[1, 0]


#Nodos puros: 3

+ Determinar las reglas para los nuevos casos.
+ Indicar la regla del negocio a la cual pertenence el primer individuo de la base de datos (In App Pure Score)
+ 	Indicar la regla del negocio a la cual pertenence el segundo individuo de la base de datos (Travel Plan Score)



In [32]:
def regla_para_fila(modelo, X_row, feature_names):
    node_indicator = modelo.decision_path(X_row.values.reshape(1, -1))
    idxs = node_indicator.indices
    reglas = []
    tree = modelo.tree_

    for i in idxs:
        left = tree.children_left[i]
        right = tree.children_right[i]
        if left == -1 and right == -1:
            continue
        feat_id = tree.feature[i]
        thr = tree.threshold[i]
        feat = feature_names[feat_id]
        valor = float(X_row.iloc[0, feat_id])
        if valor <= thr:
            reglas.append(f"{feat} <= {thr:.2f}")
        else:
            reglas.append(f"{feat} > {thr:.2f}")
    return " AND ".join(reglas)

print("Reglas de negocio de cada jugador :")
for i in range(len(Prueba_2_jugadores)):
    r = regla_para_fila(mar, Prueba_2_jugadores[["Age","Income","Years","Hours"]].iloc[[i]], list(XD.columns))
    print(f"Jugador {i+1}: {r}")

Reglas de negocio de cada jugador :
Jugador 1: Income > 63.50 AND Years > 1.50 AND Years <= 5.50 AND Income > 76.50
Jugador 2: Income > 63.50 AND Years <= 1.50 AND Age > 23.00



/usr/local/lib/python3.11/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but DecisionTreeClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but DecisionTreeClassifier was fitted with feature names
  warnings.warn(


#Reglas de negocio de cada jugador :
##Jugador 1: Income > 63.50 AND Years > 1.50 AND Years <= 5.50 AND Income > 76.50
##Jugador 2: Income > 63.50 AND Years <= 1.50 AND Age > 23.00

+	Determinar el porcentaje que una persona utilice la tarjeta de crédito para la siguiente regla SI Income	>63.5 AND Years>1.5 AND Years>5.5 AND Income<=137.5.

In [36]:
if all(col in XDB.columns for col in ["Income", "Years"]):
    cond = (XDB["Income"] > 63.5) & (XDB["Years"] > 1.5) & (XDB["Years"] <= 5.5) & (XDB["Income"] <= 137.5)
    sub = XDB[cond]
    print(f"Muestras que cumplen la regla: {len(sub)}")

    # Si existe CreditCard en el subconjunto, calculamos % con tarjeta
    if "CreditCard" in sub.columns and len(sub) > 0:
        pct_credit = (sub["CreditCard"] == 1).mean() * 100
        print(f"% con tarjeta de crédito: {pct_credit:.2f}%")

Muestras que cumplen la regla: 135
% con tarjeta de crédito: 50.37%


#Muestras que cumplen la regla: 135
#% con tarjeta de crédito: 50.37%

#Analisis de Resultados
El modelo alcanza una Exactitud general del 84.75%. Esta cifra, aunque útil como medida global, debe analizarse junto con otras métricas para comprender el comportamiento del modelo en detalle. Una alta exactitud indica que la mayoría de las predicciones, tanto de compra como de no compra, son correctas.

Un punto fuerte notable del modelo es su alta Sensibilidad del 94.79%. Esto es particularmente valioso desde una perspectiva de negocio. Una sensibilidad elevada significa que el modelo es altamente efectivo en la identificación de la gran mayoría de los jugadores que realmente realizarán una compra. Esto se traduce en una excelente capacidad para detectar el "verdadero positivo", minimizando los "falsos negativos" (solo 11 en este caso), es decir, los compradores reales que el modelo no identificó. Esta alta sensibilidad permite dirigir eficazmente los esfuerzos y recursos de marketing hacia el segmento de jugadores con mayor potencial de monetización.

Sin embargo, la Especificidad del 73.54% sugiere un área de mejora. La especificidad mide la capacidad del modelo para identificar correctamente a los jugadores que no realizarán una compra (Verdaderos Negativos). Una especificidad moderada, como en este caso, implica la presencia de un número significativo de Falsos Positivos (50). Estos son jugadores a los que el modelo predice erróneamente que comprarán, cuando en realidad no lo harán. Desde una perspectiva de negocio, los falsos positivos pueden resultar en costos ineficientes, como el gasto en campañas de marketing o promociones dirigidas a individuos que no tienen intención de comprar. Mitigar los falsos positivos puede optimizar la rentabilidad de las estrategias de monetización.

La Precisión del 80% es aceptable y complementa la Sensibilidad. Nos indica que, de todas las veces que el modelo predijo que un jugador compraría, el 80% de esas predicciones fueron correctas. Combinada con la alta Sensibilidad, sugiere que el modelo es bueno en encontrar a los compradores y, cuando predice una compra, hay una probabilidad buena de que sea acertado.

La Predictividad Negativa del 92.67% es notablemente alta. Esto significa que cuando el modelo predice que un jugador no realizará una compra, podemos tener una alta confianza en esa predicción. Esto es útil para identificar jugadores a los que quizás no valga la pena dirigir ciertos tipos de campañas de marketing o a quienes se les puede ofrecer experiencias diferentes dentro de la aplicación.

Ademas se puede observar que en el arbol hay 3 nodos puros y  el modelo es fuerte identificando a los compradores potenciales (alta Sensibilidad), sin embargo la especificidad de 74% nos indica que tiene un buen comportamiento para predecir no compradores.